<a href="https://colab.research.google.com/github/DuaaMahar5/FlyRank-internship-ml/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DuaaMahar5/FlyRank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I'm using Logistic Regression. My target is is_declining_label, a yes/no outcome. Logistic Regression outputs a probability for each page, and I sort pages by that probability to build my ranked queue. It's simple and readable, like my baseline was. I'll only try a more complex model later if this one clearly isn't enough

In [41]:
# This cell is for CODE (numbers, a query, a check).


In [42]:

!git clone https://github.com/DuaaMahar5/FlyRank-internship-ml

!ls -F

fatal: destination path 'FlyRank-internship-ml' already exists and is not an empty directory.
FlyRank-internship-ml/	sample_data/


### Data Loading

Before we can perform any operations, we need to load the data into a pandas DataFrame. Replace the placeholder below with the actual path to your dataset.

In [43]:
import pandas as pd
df = pd.read_csv('FlyRank-internship-ml/data/raw/content_refresh_anonymized.csv')

# Display the first few rows to confirm it loaded correctly
display(df.head())

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Each content_id appears exactly once in the data, so there's no risk of the same page leaking between train and test.

 I used a random 80/20 split, stratified on is_declining_label so the decline rate stays similar in both sets. I fixed random_state=42 so the split is reproducible.

In [44]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Convert the 'trend_direction' column to a binary target variable 'is_declining_label'
# Assuming 'down' implies declining (1), and other values imply not declining (0)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("Each content_id appears once:", df["content_id"].nunique() == len(df))

# --- Baseline score calculation (moved here) ---
def check_stale(row):
    if row["days_since_last_update"] > 90:
        return True
    else:
        return False

def check_ctr_gap(row):
    if row["avg_position"] > 0 and row["avg_position"] <= 20 and row["ctr"] < 0.5:
        return True
    else:
        return False

def score_row(row):
    is_stale = check_stale(row)
    has_ctr_gap = check_ctr_gap(row)

    score = 0
    reason = "NONE"

    if is_stale:
        score = score + 2
        reason = "STALE"

    if has_ctr_gap:
        score = score + 3
        reason = "CTR_GAP"

    if score >= 4:
        action = "priority_review"
    elif score >= 2:
        action = "monitor"
    else:
        action = "no_action"

    return pd.Series([score, reason, action])

df[["baseline_score", "reason_code", "action_label"]] = df.apply(score_row, axis=1)
print(f"'baseline_score' in df.columns: {'baseline_score' in df.columns}")
# --- End of baseline score calculation ---

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["is_declining_label"], random_state=42
)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train decline rate:", train_df["is_declining_label"].mean())
print("Test decline rate:", test_df["is_declining_label"].mean())

Each content_id appears once: True
'baseline_score' in df.columns: True
Train rows: 24000
Test rows: 6000
Train decline rate: 0.5420833333333334
Test decline rate: 0.542


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [45]:
print(test_df.columns.tolist())

# The functions check_stale, check_ctr_gap, score_row and their application to df
# have been moved to an earlier cell (OpXe7Qhb1aOW) to ensure baseline_score
# is present in train_df and test_df after the split.

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import pandas as pd

features = ["days_since_last_update", "avg_position", "ctr", "search_volume"]

X_train = train_df[features].fillna(0)
y_train = train_df["is_declining_label"]
X_test = test_df[features].fillna(0)
y_test = test_df["is_declining_label"]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)

test_df["model_score"] = model.predict_proba(X_test_s)[:, 1]

def precision_at_k(frame, col, k):
    top = frame.sort_values(col, ascending=False).head(k)
    return top["is_declining_label"].mean()

results = pd.DataFrame([
    {"model": "Baseline",
     "precision@50": precision_at_k(test_df, "baseline_score", 50),
     "precision@100": precision_at_k(test_df, "baseline_score", 100),
     "auc": roc_auc_score(y_test, test_df["baseline_score"])},
    {"model": "Logistic Regression",
     "precision@50": precision_at_k(test_df, "model_score", 50),
     "precision@100": precision_at_k(test_df, "model_score", 100),
     "auc": roc_auc_score(y_test, test_df["model_score"])}
])
results

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label', 'baseline_score', 'reason_code', 'action_label']


,model,precision@50,precision@100,auc
0,Baseline,0.6,0.63,0.584487
1,Logistic Regression,0.5,0.61,0.530994


My baseline actually beat Logistic Regression on every metric. Precision@50 was 0.60 for baseline vs 0.50 for the model. Precision@100 was 0.63 vs 0.61. AUC was 0.58 vs 0.53.

So my simple rule from Week 4 did better at pulling truly declining pages to the top than a trained model using the same four features. This tells me the features alone (days_since_last_update, avg_position, ctr, search_volume) don't carry much more signal than my hand-picked thresholds already captured.

Both are barely above random guessing, my base decline rate was around 0.54, so an AUC of 0.53 to 0.58 is weak either way.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [46]:
coefs = pd.DataFrame({"feature": features, "weight": model.coef_[0]})
display(coefs.sort_values("weight", ascending=False))

test_df["wrong"] = (test_df["is_declining_label"] == 1) & (test_df["model_score"] < 0.5)
wrong_cases = test_df[test_df["wrong"]].sort_values("model_score").head(3)
display(wrong_cases[["content_id", "model_score", "is_declining_label"] + features])

,feature,weight
0,days_since_last_update,0.179101
3,search_volume,-0.025489
1,avg_position,-0.086606
2,ctr,-0.186515


,content_id,model_score,is_declining_label,days_since_last_update,avg_position,ctr,search_volume
23748,content_c0af3d6f9dd3,0.060337,1,20,3.5,50.00,NaN
15856,content_dfce82404813,0.134801,1,8,9.0,33.33,0.0
27614,content_af6a0a77f5d6,0.216426,1,20,2.5,25.00,NaN


The model leans most on ctr and days_since_last_update, which matches my Week-4 signal audit, both were the two signals I built my rule on. avg_position has a small, backwards-looking weight, but this isn't leakage, it matches what I already found in Week 4: the worst-position pages actually had the lowest decline rate, not the highest. search_volume barely matters, close to zero weight.

Looking at the 3 cases the model missed most confidently: all three are pages that were updated recently (8 to 20 days ago, well under my 90-day staleness cutoff) and had good CTR and good average position, yet still declined.

 My baseline rule would have scored all three as no_action or monitor at best, since neither STALE nor CTR_GAP would fire. This shows a real blind spot: my features assume decline shows up as staleness or a CTR gap, but these pages declined without either warning sign showing up first. That's a genuinely hard case, something must be driving the decline that isn't in my four features at all.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.